In [28]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Any, Dict, List, Optional
import os
from pydantic import BaseModel
import json
from pathlib import Path

In [2]:
load_dotenv(r"D:\python_projects\pythonProject\agents\agents\.env", override=True)

True

In [5]:
open_router_api = os.getenv("OPEN_ROUTER")
print(open_router_api[:5])

sk-or


In [6]:
OPEN_ROUTER_BASE_URL = "https://openrouter.ai/api/v1"
open_router_client = AsyncOpenAI(base_url=OPEN_ROUTER_BASE_URL,api_key=open_router_api)
open_router_model = OpenAIChatCompletionsModel(model="nvidia/nemotron-3-nano-30b-a3b:free",openai_client=open_router_client)

In [21]:
instruction1 = """You are PresentationAlignmentCoach, an expert at evaluating how well a speaker narration matches what appears on their slides using OCR text, slide descriptions, layout descriptions, and speech transcripts, supported by quantitative metrics.

You will receive a single JSON presentation report containing:
- Overall speech metrics
- Overall slide metrics
- Derived metrics
- Per-slide data including timing, OCR text, slide description, layout description, speech text, speech metrics, and content alignment scores.

Your primary objective is to evaluate and report on speech slide alignment. Layout and delivery quality are secondary and should be mentioned only when clearly strong or clearly problematic.

OUTPUT FORMAT (STRICT JSON ONLY):
Return ONLY valid JSON (no markdown) with this exact shape:

{
  "overall_statistics": {
    "slides_count": <number or null>,
    "average_words_per_slide": <number or null>,
    "average_alignment_similarity": <number or null>,
    "global_speech_wpm": <number or null>,
    "global_intelligibility": <number or null>,
    "global_noise_fraction": <number or null>,
    "total_fillers": <number or null>
  },
  "high_level_summary": [ "<bullet>", "<bullet>", "... up to 5" ],
  "slides": [
    {
      "slide_id": <number or string>,
      "start_time": <number or null>,
      "end_time": <number or null>,
      "ocr_excerpt": "<string>",
      "description_excerpt": "<string>",
      "speech_excerpt": "<string>",
      "metrics": {
        "jaccard": <number or null>,
        "edit_ratio": <number or null>,
        "wpm": <number or null>,
        "intelligibility": <number or null>,
        "noise_fraction": <number or null>,
        "fillers": <number or null>,
        "speech_coverage_ratio": <number or null>
      },
      "alignment_assessment": "well_aligned" | "partially_aligned" | "not_aligned",
      "evidence": [ "<evidence line>", "<evidence line>" ],
      "layout_note": "<string or empty>",
      "recommendations": [ "<action>", "<action>", "... up to 5" ]
    }
  ],
  "patterns": [ "<pattern>", "<pattern>" ],
  "action_plan": [ "<action>", "<action>", "... 5 to 8" ]
}

Rules:
- If missing, use null (numbers) or empty string/list.
- Keep excerpts short. Do not paste full OCR/transcript.
- Evidence must include concrete keywords/phrases from BOTH speech and slide content.
- layout_note should usually be empty unless clearly good or clearly problematic.
"""

In [22]:
description = (
    "Analyzes a presentation report JSON and returns strict JSON slide-by-slide feedback focused on "
    "speech–slide alignment using OCR and slide descriptions, with evidence and metrics."
)

In [23]:
per_slide_agent = Agent(name="per_slide_feedback", instructions=instruction1, model=open_router_model)
per_slide_tool = per_slide_agent.as_tool(tool_name="per_slide_agent", tool_description=description)

In [ ]:
@function_tool
def format_alignment_report_markdown(report_json: str) -> str:
    """
    Convert the strict JSON output from per_slide_agent into clean, structured Markdown.
    Input: report_json (string) = JSON text returned by per_slide_agent.
    Output: Markdown string.
    """
    def _safe(v: Any, default: str = "not provided") -> str:
        if v is None:
            return default
        if isinstance(v, float):
            return f"{v:.3f}".rstrip("0").rstrip(".")
        return str(v)

    try:
        report: Dict[str, Any] = json.loads(report_json)
    except Exception:
        # If tool gets non-JSON, return it as-is so you can debug quickly
        return f"# Formatter Error\n\nCould not parse JSON.\n\nRaw output:\n\n```\n{report_json}\n```"

    out: List[str] = []
    out.append("# 📊 Presentation Alignment Report\n")

    stats = report.get("overall_statistics", {}) or {}
    out.append("## Overall Statistics")
    out.append(f"- **Slides count**: {_safe(stats.get('slides_count'))}")
    out.append(f"- **Average words per slide**: {_safe(stats.get('average_words_per_slide'))}")
    out.append(f"- **Average alignment similarity**: {_safe(stats.get('average_alignment_similarity'))}")
    out.append(f"- **Global speech WPM**: {_safe(stats.get('global_speech_wpm'))}")
    out.append(f"- **Global intelligibility**: {_safe(stats.get('global_intelligibility'))}")
    out.append(f"- **Global noise fraction**: {_safe(stats.get('global_noise_fraction'))}")
    out.append(f"- **Total fillers**: {_safe(stats.get('total_fillers'))}")
    out.append("")

    out.append("## High-Level Alignment Summary")
    summary = report.get("high_level_summary", []) or []
    if summary:
        for b in summary[:5]:
            out.append(f"- {b}")
    else:
        out.append("- not provided")
    out.append("")

    out.append("## Per-Slide Analysis & Recommendations")
    slides = report.get("slides", []) or []
    if not slides:
        out.append("\n_No slide data provided._\n")
    else:
        for s in slides:
            sid = _safe(s.get("slide_id"), default="unknown")
            st = _safe(s.get("start_time"))
            et = _safe(s.get("end_time"))

            out.append("\n---\n")
            out.append(f"### Slide {sid} ({st}s–{et}s)\n")

            out.append("**Reference**")
            out.append(f"- **OCR excerpt:** {(s.get('ocr_excerpt') or '').strip() or 'not provided'}")
            out.append(f"- **Description excerpt:** {(s.get('description_excerpt') or '').strip() or 'not provided'}")
            out.append(f"- **Speech excerpt:** {(s.get('speech_excerpt') or '').strip() or 'not provided'}")
            out.append("")

            m = s.get("metrics", {}) or {}
            out.append("**Metrics**")
            out.append(f"- Alignment: jaccard={_safe(m.get('jaccard'))}, edit_ratio={_safe(m.get('edit_ratio'))}")
            out.append(
                f"- Speech: wpm={_safe(m.get('wpm'))}, intelligibility={_safe(m.get('intelligibility'))}, "
                f"noise_fraction={_safe(m.get('noise_fraction'))}, fillers={_safe(m.get('fillers'))}, "
                f"coverage={_safe(m.get('speech_coverage_ratio'))}"
            )
            out.append("")

            out.append("**Alignment Assessment**")
            out.append(f"- {(s.get('alignment_assessment') or 'not provided')}")
            out.append("")

            out.append("**Evidence**")
            ev = s.get("evidence", []) or []
            if ev:
                for e in ev:
                    out.append(f"- {e}")
            else:
                out.append("- not provided")
            out.append("")

            ln = (s.get("layout_note") or "").strip()
            if ln:
                out.append("**Layout Note**")
                out.append(f"- {ln}")
                out.append("")

            out.append("**Recommendations**")
            recs = s.get("recommendations", []) or []
            if recs:
                for r in recs[:5]:
                    out.append(f"- {r}")
            else:
                out.append("- not provided")

    out.append("\n---\n")
    out.append("## Key Patterns Detected")
    patterns = report.get("patterns", []) or []
    if patterns:
        for p in patterns:
            out.append(f"- {p}")
    else:
        out.append("- not provided")

    out.append("\n## Actionable Improvement Plan")
    plan = report.get("action_plan", []) or []
    if plan:
        for a in plan[:8]:
            out.append(f"- {a}")
    else:
        out.append("- not provided")

    out.append("")
    return "\n".join(out)


In [25]:
instruction_orchestrator = """
You are an orchestrator.

Steps:
1) Call tool `per_slide_agent` with the provided presentation report JSON.
2) Take the tool output (STRICT JSON text) and pass it as-is to `format_alignment_report_markdown`.
3) Return ONLY the Markdown from `format_alignment_report_markdown`.
"""

In [26]:
report_agent = Agent(
    name="report_agent",
    instructions=instruction_orchestrator,
    model=open_router_model,
    tools=[per_slide_tool, format_alignment_report_markdown],
)

In [29]:
presentation_json = json.loads(Path("content_report_payload.json").read_text(encoding="utf-8"))

In [30]:
message = "Generate the final Markdown report for this presentation JSON:\n\n" + json.dumps(
    presentation_json, ensure_ascii=False
)

with trace("Slide_feedback_system"):
    result = await Runner.run(report_agent, message)

In [31]:
final_md = result.final_output
Path("final_report.md").write_text(final_md, encoding="utf-8")

print("Saved: final_report.md")
print("Chars:", len(final_md))

Saved: final_report.md
Chars: 3109
